# NB06: Blocked Reactions Analysis

For each reference model, run FVA with `FullThermoPkg` thermodynamic constraints
using old (Marvin) vs new (OPAM2) deltaG values. Count newly blocked/unblocked
reactions and analyze which metabolic subsystems are most affected.

**FullThermoPkg** applies: `constant = deltag * 4.184 + charge * Faraday * membrane_potential`

A reaction is **blocked** when FVA gives min=0 and max=0.

In [ ]:
import sys
import json
from pathlib import Path

import cobra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

KBUTILLIB_ROOT = Path('/global_share/KBaseUtilities/KBUtilLib')
sys.path.insert(0, str(KBUTILLIB_ROOT / 'src'))

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'
MSDB_ROOT = Path('/tmp/ModelSEEDDatabase')

from kbutillib import MSFBAUtils, ThermoUtils

## 1. Load models and deltaG data

In [ ]:
# Load deltaG comparison data from NB03
df_dg = pd.read_csv(DATA_DIR / 'dg_comparison.tsv', sep='\t')
old_dg_map = dict(zip(df_dg['rxn_id'], df_dg['old_dg']))
new_dg_map = dict(zip(df_dg['rxn_id'], df_dg['new_dg']))
print(f'Loaded deltaG data for {len(df_dg):,} reactions')

# Load saved models
model_files = sorted(DATA_DIR.glob('model_*.json'))
models = {}
for f in model_files:
    org_id = f.stem.replace('model_', '')
    models[org_id] = cobra.io.load_json_model(str(f))
    print(f'Loaded {org_id}: {len(models[org_id].reactions)} reactions')

## 2. Run FVA with thermodynamic constraints

For each model, apply FullThermoPkg with old deltaG values and new deltaG values,
then run FVA to classify reactions.

In [ ]:
from modelseedpy.fbapkg import FullThermoPkg
from modelseedpy.core.msmodelutl import MSModelUtil

def classify_fva(fva_results):
    """Classify reactions from FVA results into Blocked/Positive/Negative/Variable."""
    classes = {}
    for rxn_id, vals in fva_results.items():
        mn = vals.get('MIN', 0)
        mx = vals.get('MAX', 0)
        if mn is None:
            mn = 0
        if mx is None:
            mx = 0
        tol = 1e-9
        if abs(mn) < tol and abs(mx) < tol:
            classes[rxn_id] = 'Blocked'
        elif mn >= -tol and mx > tol:
            classes[rxn_id] = 'Forward'
        elif mn < -tol and mx <= tol:
            classes[rxn_id] = 'Reverse'
        else:
            classes[rxn_id] = 'Reversible'
    return classes

def run_thermo_fva(model, dg_map):
    """Apply thermodynamic constraints and run FVA."""
    mdlutl = MSModelUtil.get(model.copy())

    # Apply thermodynamic package
    thermo_pkg = FullThermoPkg(mdlutl)
    thermo_pkg.build_package({
        'default_min_uptake': -100,
        'default_max_uptake': 0,
    })

    # Run FVA
    fba_utils = MSFBAUtils()
    fva_results = fba_utils.run_fva(mdlutl, fraction_of_optimum=0.0)
    return classify_fva(fva_results)

In [ ]:
all_results = []

for org_id, model in models.items():
    print(f'\n=== {org_id} ===')

    # Without thermo constraints (baseline)
    fba_utils = MSFBAUtils()
    mdlutl_base = MSModelUtil.get(model.copy())
    fva_base = fba_utils.run_fva(mdlutl_base, fraction_of_optimum=0.0)
    classes_base = classify_fva(fva_base)

    # With old (Marvin) thermo constraints
    # Note: FullThermoPkg reads deltaG from ModelSEEDBiochem or model notes
    # For the old/new comparison, we need to point to old vs new DB
    classes_old = classes_base  # Placeholder: will need actual thermo FVA
    classes_new = classes_base  # Placeholder: will need actual thermo FVA

    # Compare
    common = set(classes_old.keys()) & set(classes_new.keys())
    newly_blocked = sum(1 for r in common if classes_old[r] != 'Blocked' and classes_new[r] == 'Blocked')
    newly_unblocked = sum(1 for r in common if classes_old[r] == 'Blocked' and classes_new[r] != 'Blocked')
    direction_changed = sum(1 for r in common if classes_old[r] != classes_new[r] and
                           classes_old[r] != 'Blocked' and classes_new[r] != 'Blocked')

    n_blocked_base = sum(1 for c in classes_base.values() if c == 'Blocked')
    n_blocked_old = sum(1 for c in classes_old.values() if c == 'Blocked')
    n_blocked_new = sum(1 for c in classes_new.values() if c == 'Blocked')

    print(f'  Reactions: {len(common)}')
    print(f'  Blocked (no thermo): {n_blocked_base}')
    print(f'  Blocked (old thermo): {n_blocked_old}')
    print(f'  Blocked (new thermo): {n_blocked_new}')
    print(f'  Newly blocked: {newly_blocked}')
    print(f'  Newly unblocked: {newly_unblocked}')
    print(f'  Direction changed: {direction_changed}')

    all_results.append({
        'organism': org_id,
        'total_reactions': len(common),
        'blocked_no_thermo': n_blocked_base,
        'blocked_old_thermo': n_blocked_old,
        'blocked_new_thermo': n_blocked_new,
        'newly_blocked': newly_blocked,
        'newly_unblocked': newly_unblocked,
        'direction_changed': direction_changed,
    })

## 3. Save results

In [ ]:
if all_results:
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(DATA_DIR / 'blocked_reactions_diff.tsv', sep='\t', index=False)
    print(df_results.to_string(index=False))

## 4. Visualize blocked reaction changes

In [ ]:
if all_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(all_results))
    labels = [r['organism'] for r in all_results]
    ax.bar([i - 0.2 for i in x], [r['blocked_old_thermo'] for r in all_results],
           0.4, label='Old (Marvin)', color='tab:blue', alpha=0.7)
    ax.bar([i + 0.2 for i in x], [r['blocked_new_thermo'] for r in all_results],
           0.4, label='New (OPAM2)', color='tab:orange', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Number of blocked reactions')
    ax.set_title('Blocked reactions: Marvin vs OPAM2 thermodynamics')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'blocked_reactions_comparison.png', dpi=150)
    plt.show()